# β-VAE: Disentangled Generative Modeling on MNIST

**Author:** Chris Schmidt  
**Date:** April 2026  

This notebook implements a β-Variational Autoencoder (β-VAE) from scratch in PyTorch and explores
how the β hyperparameter controls the trade-off between reconstruction fidelity and latent
disentanglement.

## Contents
1. [Background & Motivation](#1-background--motivation)
2. [Setup & Data Loading](#2-setup--data-loading)
3. [Model Architecture](#3-model-architecture)
4. [Training a Standard VAE (β=1)](#4-training-a-standard-vae-β1)
5. [Effect of β on Disentanglement](#5-effect-of-β-on-disentanglement)
6. [Latent Space Visualization](#6-latent-space-visualization)
7. [Latent Interpolation](#7-latent-interpolation)
8. [Generation Quality](#8-generation-quality)
9. [Key Findings & Takeaways](#9-key-findings--takeaways)

## 1. Background & Motivation

A **Variational Autoencoder (VAE)** learns a generative model $p_\theta(x)$ by maximizing the
evidence lower bound (ELBO):

$$\mathcal{L}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - \text{KL}(q_\phi(z|x) \| p(z))$$

The **β-VAE** (Higgins et al., 2017) introduces a coefficient β on the KL term:

$$\mathcal{L}_\beta = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - \beta \cdot \text{KL}(q_\phi(z|x) \| p(z))$$

- **β = 1**: Standard VAE  
- **β > 1**: Stronger pressure toward the prior → more disentangled latent codes, but blurrier reconstructions  
- **β < 1**: Weaker regularization → sharper reconstructions, less structured latent space

This study trains β-VAE models across β ∈ {0.5, 1.0, 4.0, 10.0} on MNIST and compares
reconstruction quality, generation quality, and latent space structure.

## 2. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from src.models import BetaVAE
from src.training import train_model, evaluate, vae_loss, TrainingMetrics
from src.visualization import (
    plot_loss_curves,
    plot_reconstructions,
    plot_generations,
    plot_latent_space,
    plot_latent_interpolation,
    plot_beta_comparison,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVIDENCE_DIR = Path("../evidence")
EVIDENCE_DIR.mkdir(exist_ok=True)

print(f"PyTorch {torch.__version__} | Device: {DEVICE}")

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = datasets.MNIST(root="../data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="../data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Training: {len(train_dataset):,} images | Test: {len(test_dataset):,} images")

## 3. Model Architecture

The encoder uses two convolutional layers (stride 2 for downsampling) followed by FC layers
that output μ and log σ² for the latent distribution. The decoder mirrors this with transposed
convolutions.

| Component | Layer | Output Shape |
|-----------|-------|--------------|
| **Encoder** | Conv2d(1→32, 3×3, s=2) | 32×14×14 |
| | Conv2d(32→64, 3×3, s=2) | 64×7×7 |
| | FC(3136→256) | 256 |
| | FC(256→latent_dim) × 2 | μ, log σ² |
| **Decoder** | FC(latent_dim→256) | 256 |
| | FC(256→3136) → reshape | 64×7×7 |
| | ConvT2d(64→32, 3×3, s=2) | 32×14×14 |
| | ConvT2d(32→1, 3×3, s=2) | 1×28×28 |

In [ ]:
# Inspect architecture
LATENT_DIM = 10
model_demo = BetaVAE(latent_dim=LATENT_DIM, beta=1.0)
total_params = sum(p.numel() for p in model_demo.parameters())
print(f"Latent dim: {LATENT_DIM}")
print(f"Total parameters: {total_params:,}")
print()
print(model_demo)

## 4. Training a Standard VAE (β=1)

First, we train a baseline VAE with β=1 to establish reconstruction quality.

In [ ]:
EPOCHS = 15

model_b1 = BetaVAE(latent_dim=LATENT_DIM, beta=1.0).to(DEVICE)
metrics_b1 = train_model(model_b1, train_loader, epochs=EPOCHS, lr=1e-3, device=DEVICE)

test_total, test_recon, test_kl = evaluate(model_b1, test_loader, device=DEVICE)
print(f"\nTest loss (β=1): total={test_total:.2f}, recon={test_recon:.2f}, KL={test_kl:.2f}")

In [ ]:
fig = plot_loss_curves(metrics_b1, title="Standard VAE (β=1) Training",
                       save_path=EVIDENCE_DIR / "loss_curves_beta1.png")
plt.show()

In [ ]:
# Get a batch for reconstruction visualization
test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(DEVICE)

fig = plot_reconstructions(model_b1, test_images, n_show=10,
                           save_path=EVIDENCE_DIR / "reconstructions_beta1.png")
plt.show()

## 5. Effect of β on Disentanglement

We now train models with β ∈ {0.5, 4.0, 10.0} and compare against the β=1 baseline.

In [ ]:
beta_values = [0.5, 4.0, 10.0]
models = {1.0: model_b1}
all_metrics = {1.0: metrics_b1}

for beta in beta_values:
    print(f"\n--- Training β={beta} ---")
    model = BetaVAE(latent_dim=LATENT_DIM, beta=beta).to(DEVICE)
    metrics = train_model(model, train_loader, epochs=EPOCHS, lr=1e-3, device=DEVICE)
    models[beta] = model
    all_metrics[beta] = metrics
    
    t_total, t_recon, t_kl = evaluate(model, test_loader, device=DEVICE)
    print(f"  Test: total={t_total:.2f}, recon={t_recon:.2f}, KL={t_kl:.2f}")

print("\nAll models trained.")

In [ ]:
fig = plot_beta_comparison(all_metrics, save_path=EVIDENCE_DIR / "beta_comparison.png")
plt.show()

In [ ]:
# Side-by-side reconstructions across β values
fig, axes = plt.subplots(len(models) + 1, 10, figsize=(15, 2 * (len(models) + 1)))

sample_images = test_images[:10]
for i in range(10):
    axes[0, i].imshow(sample_images[i, 0].cpu().numpy(), cmap="gray")
    axes[0, i].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=9)

for row, beta in enumerate(sorted(models.keys()), 1):
    recon = models[beta].reconstruct(sample_images).cpu().numpy()
    for i in range(10):
        axes[row, i].imshow(recon[i, 0], cmap="gray")
        axes[row, i].axis("off")
    axes[row, 0].set_ylabel(f"β={beta}", fontsize=9)

plt.suptitle("Reconstruction Quality vs β", fontsize=14)
plt.tight_layout()
fig.savefig(EVIDENCE_DIR / "reconstruction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Latent Space Visualization

Projecting encoded test images onto the first two latent dimensions reveals how β affects
latent structure. Higher β should produce tighter, more Gaussian-like clusters.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, beta in zip(axes, sorted(models.keys())):
    model = models[beta]
    model.eval()
    all_mu, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            mu, _ = model.encoder(imgs.to(DEVICE))
            all_mu.append(mu.cpu())
            all_labels.append(lbls)
    mu_cat = torch.cat(all_mu).numpy()
    labels_cat = torch.cat(all_labels).numpy()
    scatter = ax.scatter(mu_cat[:, 0], mu_cat[:, 1], c=labels_cat, cmap="tab10", s=2, alpha=0.5)
    ax.set_title(f"β={beta}")
    ax.set_xlabel("z₁")
    ax.set_ylabel("z₂")

plt.colorbar(scatter, ax=axes[-1], label="Digit")
plt.suptitle("Latent Space Structure Across β Values", fontsize=14)
plt.tight_layout()
fig.savefig(EVIDENCE_DIR / "latent_space_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Latent Interpolation

Linear interpolation between two encoded digits shows the smoothness of the learned manifold.
More disentangled models (higher β) should produce smoother, more interpretable transitions.

In [ ]:
# Find a '3' and a '7' in the test set for interpolation
idx_3 = next(i for i, (_, l) in enumerate(test_dataset) if l == 3)
idx_7 = next(i for i, (_, l) in enumerate(test_dataset) if l == 7)

img_3 = test_dataset[idx_3][0].unsqueeze(0).to(DEVICE)
img_7 = test_dataset[idx_7][0].unsqueeze(0).to(DEVICE)

N_STEPS = 12
fig, axes = plt.subplots(len(models), N_STEPS, figsize=(N_STEPS * 1.2, len(models) * 1.5))

for row, beta in enumerate(sorted(models.keys())):
    model = models[beta]
    model.eval()
    with torch.no_grad():
        z_3, _ = model.encoder(img_3)
        z_7, _ = model.encoder(img_7)
    
    alphas = torch.linspace(0, 1, N_STEPS).unsqueeze(1).to(DEVICE)
    z_interp = z_3 * (1 - alphas) + z_7 * alphas
    
    with torch.no_grad():
        decoded = model.decoder(z_interp).cpu().numpy()
    
    for i in range(N_STEPS):
        axes[row, i].imshow(decoded[i, 0], cmap="gray")
        axes[row, i].axis("off")
    axes[row, 0].set_ylabel(f"β={beta}", fontsize=9)

plt.suptitle("Latent Interpolation: 3 → 7", fontsize=14)
plt.tight_layout()
fig.savefig(EVIDENCE_DIR / "latent_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Generation Quality

Samples drawn from the prior $z \sim \mathcal{N}(0, I)$ and decoded. Higher β typically
produces more recognizable but less diverse samples.

In [ ]:
fig, axes = plt.subplots(len(models), 16, figsize=(20, len(models) * 1.5))

torch.manual_seed(42)
z_shared = torch.randn(16, LATENT_DIM).to(DEVICE)

for row, beta in enumerate(sorted(models.keys())):
    model = models[beta]
    model.eval()
    with torch.no_grad():
        samples = model.decoder(z_shared).cpu().numpy()
    for i in range(16):
        axes[row, i].imshow(samples[i, 0], cmap="gray")
        axes[row, i].axis("off")
    axes[row, 0].set_ylabel(f"β={beta}", fontsize=9)

plt.suptitle("Generated Samples from Shared Latent Vectors", fontsize=14)
plt.tight_layout()
fig.savefig(EVIDENCE_DIR / "generated_samples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Summary table of test metrics
import pandas as pd

rows = []
for beta in sorted(models.keys()):
    t_total, t_recon, t_kl = evaluate(models[beta], test_loader, device=DEVICE)
    rows.append({"β": beta, "Total Loss": f"{t_total:.2f}", "Recon Loss": f"{t_recon:.2f}", "KL Div": f"{t_kl:.2f}"})

df = pd.DataFrame(rows)
print("\n=== Test Metrics Summary ===")
print(df.to_string(index=False))

## 9. Key Findings & Takeaways

**Reconstruction–Disentanglement Trade-off:**  
As β increases, reconstruction loss rises (blurrier images) but KL divergence drops, indicating
the posterior is pushed closer to the prior — a hallmark of disentanglement.

**Latent Space Structure:**  
Higher β produces more Gaussian-shaped, overlapping clusters in z-space. This is desirable for
generation (sampling from N(0,I) lands in meaningful regions) but reduces class separability.

**Interpolation Smoothness:**  
Higher-β models show smoother digit transitions in latent interpolation, confirming that the
latent manifold is more regularly structured.

**Practical Guidance:**  
- β=1 (standard VAE) is a strong default for reconstruction-focused tasks  
- β=4 offers a good balance for disentangled representations  
- β=10 aggressively regularizes — useful for downstream tasks requiring interpretable latent factors,
  but at the cost of generation sharpness

**Implementation Notes:**  
- Reparameterization trick enables gradient flow through the stochastic sampling step  
- Convolutional architecture (vs. fully-connected) preserves spatial structure, improving both
  reconstruction and generation quality on image data  
- All code is modular (`src/models.py`, `src/training.py`, `src/visualization.py`) with full
  test coverage via `pytest`